# 🧬 LoRA Fine-Tuning for Dialogue Summarization

This notebook walks through the complete process of fine-tuning a T5 model with LoRA adapters for customer service dialogue summarization.

**What you'll learn:**
1. How the TweetSumm dataset is structured
2. How LoRA reduces trainable parameters by ~99.7%
3. How to train and evaluate the model
4. How different LoRA ranks affect performance (ablation study)

---

## 1. Setup & Configuration

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import Counter

from src.config import load_config
from src.data import load_tokenizer, load_data
from src.model import build_model, get_model_stats
from src.inference import summarize

# Use dark theme for plots
plt.style.use('dark_background')

config = load_config()
print(f'Model: {config.model_id}')
print(f'Device: {config.device}')
print(f'LoRA rank: {config.lora.r}')
print(f'Training samples: {config.n_train}')

---
## 2. Dataset Exploration

The [TweetSumm](https://huggingface.co/datasets/Andyrasika/TweetSumm-tuned) dataset contains customer service dialogues with reference summaries.

In [ ]:
from datasets import load_dataset

ds = load_dataset(config.dataset_name)
print(f'Splits: {list(ds.keys())}')
print(f'Train: {len(ds["train"])} samples')
print(f'Validation: {len(ds["validation"])} samples')
print(f'Test: {len(ds["test"])} samples')
print(f'\nColumns: {ds["train"].column_names}')

In [ ]:
# Let's look at a sample dialogue
sample = ds['train'][0]
dlg_col = 'dialogue' if 'dialogue' in ds['train'].column_names else 'conversation'

print('=' * 60)
print('DIALOGUE:')
print('=' * 60)
print(sample[dlg_col][:500])
print('\n' + '=' * 60)
print('REFERENCE SUMMARY:')
print('=' * 60)
print(sample['summary'])

In [ ]:
# Analyze dialogue and summary lengths
dlg_lengths = [len(s[dlg_col].split()) for s in ds['train']]
sum_lengths = [len(s['summary'].split()) for s in ds['train']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(dlg_lengths, bins=30, color='#818cf8', alpha=0.8, edgecolor='#1e293b')
axes[0].set_title('Dialogue Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Count')
axes[0].axvline(np.median(dlg_lengths), color='#f472b6', linestyle='--', label=f'Median: {np.median(dlg_lengths):.0f}')
axes[0].legend()

axes[1].hist(sum_lengths, bins=20, color='#34d399', alpha=0.8, edgecolor='#1e293b')
axes[1].set_title('Summary Length Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Count')
axes[1].axvline(np.median(sum_lengths), color='#f472b6', linestyle='--', label=f'Median: {np.median(sum_lengths):.0f}')
axes[1].legend()

plt.suptitle('TweetSumm Dataset — Length Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Compression ratio: {np.mean(dlg_lengths) / np.mean(sum_lengths):.1f}x')

---
## 3. Understanding LoRA

### The Problem with Full Fine-Tuning

T5-Small has **~60M parameters**. Fine-tuning all of them:
- Requires significant GPU memory
- Risks catastrophic forgetting
- Creates a full model copy for each task

### The LoRA Solution

LoRA ([Hu et al., 2021](https://arxiv.org/abs/2106.09685)) freezes the original weights and injects **small trainable matrices** into attention layers:

$$W' = W_{\text{frozen}} + \Delta W = W + B \cdot A$$

Where:
- $W \in \mathbb{R}^{d \times d}$ — original frozen weight matrix
- $B \in \mathbb{R}^{d \times r}$ — low-rank down-projection  
- $A \in \mathbb{R}^{r \times d}$ — low-rank up-projection
- $r \ll d$ — the rank (e.g., r=4 when d=512)

With **rsLoRA** scaling: $W' = W + \frac{\alpha}{\sqrt{r}} \cdot B \cdot A$

In [ ]:
# Visualize the parameter savings
model = build_model(config)
stats = get_model_stats(model)

print(f"Total parameters:     {stats['total_params']:>12,}")
print(f"Trainable parameters: {stats['trainable_params']:>12,}")
print(f"Trainable percentage: {stats['trainable_pct']:>11.2f}%")
print(f"Model size:           {stats['model_size_mb']:>10.1f} MB")

In [ ]:
# Parameter comparison chart
fig, ax = plt.subplots(figsize=(8, 4))

frozen = stats['total_params'] - stats['trainable_params']
trainable = stats['trainable_params']

bars = ax.barh(
    ['Full Fine-Tune', 'LoRA (r=4)'],
    [stats['total_params'], trainable],
    color=['#475569', '#818cf8'],
    edgecolor='#1e293b',
    height=0.5,
)

ax.set_xlabel('Trainable Parameters', fontsize=12)
ax.set_title('Parameters Updated During Training', fontsize=14, fontweight='bold')

# Add value labels
ax.text(stats['total_params'], 0, f"  {stats['total_params']:,}", va='center', fontsize=11)
ax.text(trainable, 1, f"  {trainable:,} ({stats['trainable_pct']:.2f}%)", va='center', fontsize=11, color='#818cf8')

ax.set_xlim(0, stats['total_params'] * 1.3)
plt.tight_layout()
plt.show()

---
## 4. Training

Let's train the model and observe the loss decrease.

In [ ]:
# Load tokenizer and data
tokenizer = load_tokenizer(config)
data = load_data(config, tokenizer)

print(f'Train samples: {len(data["train"])}')
print(f'Eval samples: {len(data["eval"])}')
print(f'Test samples: {len(data["test"])}')

In [ ]:
# Pre-training baseline: what does the model generate BEFORE training?
test_sample = data['test'][0]
dlg_col = data['dialogue_column']

before = summarize(model, tokenizer, test_sample[dlg_col], config)
print('BEFORE training:')
print(f'  Generated:  {before}')
print(f'  Reference:  {test_sample["summary"]}')

In [ ]:
from src.train import create_trainer, run_training

trainer = create_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data['train'],
    eval_dataset=data['eval'],
    config=config,
)

metrics = run_training(trainer)
print(f"\nTraining loss: {metrics['train_loss']:.4f}")
print(f"Wall time: {metrics['wall_time_seconds']:.1f}s")

In [ ]:
# Post-training: what does the model generate AFTER training?
after = summarize(model, tokenizer, test_sample[dlg_col], config)
print('AFTER training:')
print(f'  Generated:  {after}')
print(f'  Reference:  {test_sample["summary"]}')
print(f'\n  BEFORE:     {before}')

---
## 5. Evaluation

Let's compute ROUGE scores to quantify the improvement.

In [ ]:
import evaluate
from tqdm import tqdm

# Generate predictions for all test samples
test_ds = data['test']
predictions = []
references = []

for sample in tqdm(test_ds, desc='Generating summaries'):
    pred = summarize(model, tokenizer, sample[dlg_col], config)
    predictions.append(pred)
    references.append(sample['summary'])

# Compute ROUGE
rouge = evaluate.load('rouge')
scores = rouge.compute(predictions=predictions, references=references)

print('\n📊 ROUGE Scores:')
for metric, value in sorted(scores.items()):
    print(f'  {metric:12s}  {value:.4f}')

In [ ]:
# Show some example predictions
print('\n' + '=' * 70)
print('  Sample Predictions')
print('=' * 70)

for i in range(min(3, len(test_ds))):
    print(f'\n--- Sample {i+1} ---')
    print(f'  Reference:  {references[i]}')
    print(f'  Generated:  {predictions[i]}')

---
## 6. Rank Ablation Study

One of the key hyperparameters in LoRA is the **rank** `r`. Let's analyze how different ranks affect performance.

We load results from a pre-computed ablation study (run via `python -m scripts.experiments`).

In [ ]:
# Load ablation results
results_dir = Path('../results')
ablation_file = results_dir / 'rank_ablation_latest.json'

if ablation_file.exists():
    with open(ablation_file) as f:
        ablation = json.load(f)
    
    results = ablation['results']
    
    print(f"Model: {ablation['model_id']}")
    print(f"Training samples: {ablation['n_train']}")
    print(f"Epochs: {ablation['epochs']}")
    print(f"Ranks tested: {ablation['ranks_tested']}")
    print()
    
    # Print table
    print(f"{'Rank':>6}  {'Params':>10}  {'%':>6}  {'ROUGE-1':>8}  {'ROUGE-2':>8}  {'ROUGE-L':>8}  {'Time':>8}")
    print('-' * 65)
    for r in results:
        print(f"r={r['rank']:<4d}  {r['model_stats']['trainable_params']:>10,}  {r['model_stats']['trainable_pct']:>5.2f}%  {r['rouge1']:>8.4f}  {r['rouge2']:>8.4f}  {r['rougeL']:>8.4f}  {r['wall_time_seconds']:>6.1f}s")
    
    best = max(results, key=lambda x: x['rougeL'])
    print(f"\n🏆 Best: r={best['rank']} (ROUGE-L={best['rougeL']:.4f})")
else:
    print('⚠️ No ablation results found. Run: python -m scripts.experiments')

In [ ]:
# Visualize ablation results
if ablation_file.exists():
    ranks = [r['rank'] for r in results]
    rouge1 = [r['rouge1'] for r in results]
    rouge2 = [r['rouge2'] for r in results]
    rougeL = [r['rougeL'] for r in results]
    trainable = [r['model_stats']['trainable_params'] for r in results]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: ROUGE vs Rank
    ax1.plot(ranks, rouge1, 'o-', color='#818cf8', linewidth=2.5, markersize=8, label='ROUGE-1')
    ax1.plot(ranks, rouge2, 's-', color='#34d399', linewidth=2.5, markersize=8, label='ROUGE-2')
    ax1.plot(ranks, rougeL, 'D-', color='#f472b6', linewidth=2.5, markersize=8, label='ROUGE-L')
    ax1.set_xlabel('LoRA Rank (r)', fontsize=12)
    ax1.set_ylabel('ROUGE Score', fontsize=12)
    ax1.set_title('ROUGE Scores vs LoRA Rank', fontsize=14, fontweight='bold')
    ax1.set_xticks(ranks)
    ax1.legend()
    ax1.grid(True, alpha=0.2)
    
    # Right: Efficiency (ROUGE-L per 100K params)
    efficiency = [rl / (tp / 100000) for rl, tp in zip(rougeL, trainable)]
    colors = ['#fbbf24' if r['rank'] == best['rank'] else '#475569' for r in results]
    ax2.bar([f'r={r}' for r in ranks], efficiency, color=colors, edgecolor='#1e293b')
    ax2.set_ylabel('ROUGE-L per 100K params', fontsize=12)
    ax2.set_title('Parameter Efficiency', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.2, axis='y')
    
    plt.suptitle('LoRA Rank Ablation Study', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

---
## 7. Conclusions

### Key Findings

1. **r=4 is optimal** for this task — the lowest rank tested achieves the highest ROUGE scores
2. **Higher ranks show diminishing returns** — the task's structure is well-captured by a rank-4 decomposition
3. **Parameter efficiency is remarkable** — 0.24% of parameters achieves strong summarization
4. **Training is fast** — ~53 seconds on Apple M4 (MPS) with 300 samples

### Why r=4 Wins

Customer service dialogues follow predictable patterns (greeting → problem → resolution → closing). The underlying semantic structure is low-dimensional, so a rank-4 decomposition captures the essential transformations without overfitting to noise that higher ranks might introduce.

### Next Steps

- Try `t5-base` for potentially higher quality (config available at `configs/t5-base.yaml`)
- Experiment with different target modules (adding `k` and `o` projections)
- Try the interactive demo: `python -m scripts.demo`